# Week 4 — Tool & Config Patterns

Covers: decorator-based tool registration (the pattern behind `@tool`-style APIs) and
reading YAML config files for multi-agent configuration.

## 1. Decorators for tool registration

In [ ]:
TOOL_REGISTRY = {}

def tool(func):
    """Registering a function as a callable tool just by decorating it — this is the
    exact mechanism behind '@tool' in agent frameworks and '@mcp.tool()' in Week 5."""
    TOOL_REGISTRY[func.__name__] = func
    return func

@tool
def lookup_order(order_id: str) -> str:
    return f"(mock) order {order_id}: shipped 2 days ago"

@tool
def check_refund_policy(product_category: str) -> str:
    return f"(mock) refund policy for {product_category}: 30-day window"

print("Registered tools:", list(TOOL_REGISTRY.keys()))

# An agent "discovering" and calling a tool dynamically by name — the pattern that lets
# an LLM's function-calling output (a tool NAME as a string) actually invoke real code.
def dispatch(tool_name: str, *args):
    if tool_name not in TOOL_REGISTRY:
        raise KeyError(f"Unknown tool requested: {tool_name}")
    return TOOL_REGISTRY[tool_name](*args)

print(dispatch("lookup_order", "ORD-9911"))
print(dispatch("check_refund_policy", "electronics"))

In [ ]:
# What happens if the model asks for a tool that doesn't exist? This must be handled,
# not assumed away — it's a direct precursor to Week 5's guardrails discussion.
try:
    dispatch("delete_all_orders", "ORD-9911")
except KeyError as e:
    print("Blocked unknown tool call:", e)

## 2. Reading YAML config — multi-agent configuration

In [ ]:
import yaml

config_text = '''
team_name: SupportPilot
escalation_threshold: 0.65
agents:
  - name: Classifier
    role: classify_ticket
  - name: Drafter
    role: draft_response
  - name: Validator
    role: check_accuracy
'''

config = yaml.safe_load(config_text)   # safe_load, never plain load, for untrusted YAML
print(config)
print()
print("Team:", config["team_name"])
print("Escalate below confidence:", config["escalation_threshold"])
for agent_cfg in config["agents"]:
    print(f"  - {agent_cfg['name']} handles {agent_cfg['role']}")

Why config files instead of hardcoding this in Python: a facilitator running SupportPilot
vs. CareRoute can swap the entire agent team topology and escalation threshold by editing
one YAML file — no code changes, no redeploy. This is the same principle Week 5's
environment/secrets management extends into production.